## Activity Occurrence Statistics

**Information Need:** Understand how activity types occur across cases and whether they occur once or repeatedly within a case.

**Motivation:** Activities can differ substantially in their occurrence across process instances (i.e., cases). Some activities may occur in nearly every case, while others occur only in specific subsets of cases. Moreover, an activity may occur once or repeatedly within a case. Understanding these characteristics helps analysts distinguish common from exceptional or less frequent process behavior and identify activities potentially associated with loops or rework.

**Approach:** For each activity type, determine its case coverage and distinguish cases with a single versus repeated occurrence.

**Output:** A case-wise occurrence summary for each activity type.

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

### Pattern execution

In [ ]:
# Count how often each activity occurs within each case
occurrence_counts = (
    event_log
    .groupby([CASE_ID, ACTIVITY])
    .size()
    .unstack(fill_value=0)
)

total_cases = event_log[CASE_ID].nunique()



# Summarize case-wise activity occurrence
cases_with_activity = (occurrence_counts > 0).sum()
cases_once = (occurrence_counts == 1).sum()
cases_multiple = (occurrence_counts > 1).sum()

activity_summary = pd.DataFrame({
    'Cases with Activity': cases_with_activity,
    'Case Coverage in %': cases_with_activity / total_cases,
    'Excactly Once': cases_once,
    #'Cases (once) in %': (cases_once / total_cases * 100).round(2),
    'Multiple Times': cases_multiple,
    #'Cases (multiple) in %': (cases_multiple / total_cases * 100).round(2),
}).sort_values('Case Coverage in %', ascending=False)

display(
    activity_summary.style
    .format({
        "Cases with Activity": "{:,.0f}",
        "Case Coverage in %": "{:.1%}",
        "Exactly Once": "{:,.0f}",
        "Multiple Times": "{:,.0f}"
    })
)